# 01 — Revised Synthetic Experiment

Runs the full outer-seed × alpha experiment comparing five methods:

| Method | Description |
|--------|-------------|
| `GLOBAL_SENTINEL` | One pooled model, sentinel encoding |
| `GLOBAL_MISSINGNESS` | One pooled model + absence indicators |
| `GLOBAL_MISSINGNESS_CDV` | One pooled model + indicators + CDV identity |
| `CDV_SEPARATE` | Separate model per retained CDV (proposed) |
| `MATCHED_RANDOM_PARTITIONS` | Placebo: random partitions matching CDV sizes |



## 1. Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'synthetic'))
import config as CFG

from cdv_utils.synthetic_dgp import get_w_cols
from helpers.runner import run_outer_seed_loop_synthetic, load_checkpoint

os.makedirs(CFG.ARTIFACTS_DIR, exist_ok=True)
os.makedirs(CFG.PLOTS_DIR, exist_ok=True)

w_cols = get_w_cols()

print('Setup complete.')
print(f'N_OUTER_SEEDS         = {CFG.N_OUTER_SEEDS}')
print(f'ALPHA_VALUES          = {CFG.ALPHA_VALUES}')
print(f'N_TRAIN               = {CFG.N_TRAIN}')
print(f'N_TEST                = {CFG.N_TEST}')
print(f'CDV_COVERAGE_THRESHOLD= {CFG.CDV_COVERAGE_THRESHOLD}')
print(f'N_RANDOM_PERMUTATIONS = {CFG.N_RANDOM_PERMUTATIONS}')
print(f'Feature columns       = {w_cols}')

## 2. Config Dict

In [ ]:
config = {
    'N_TRAIN':                CFG.N_TRAIN,
    'N_TEST':                 CFG.N_TEST,
    'CDV_COVERAGE_THRESHOLD': CFG.CDV_COVERAGE_THRESHOLD,
    'CDV_N_MIN':              CFG.CDV_N_MIN,
    'CDV_MIN_ARM_SIZE':       CFG.CDV_MIN_ARM_SIZE,
    'OVERLAP_LO':             CFG.OVERLAP_LO,
    'OVERLAP_HI':             CFG.OVERLAP_HI,
    'OVERLAP_MIN_FRACTION':   CFG.OVERLAP_MIN_FRACTION,
    'RF_N_ESTIMATORS':        CFG.RF_N_ESTIMATORS,
    'DR_FINAL_MODEL':         CFG.DR_FINAL_MODEL,
    'DR_CV':                  CFG.DR_CV,
    'N_RANDOM_PERMUTATIONS':  CFG.N_RANDOM_PERMUTATIONS,
    'N_ORACLE_CV_FOLDS':      CFG.N_ORACLE_CV_FOLDS,
    'SENTINEL_VALUE':         CFG.SENTINEL_VALUE,
}

## 3. Debug Run (2 seeds, 2 alphas)

Set `RUN_DEBUG = True` for a smoke test before the full run.

In [ ]:
RUN_DEBUG = False
FORCE_RERUN_DEBUG = True  # delete stale debug checkpoints before running

if RUN_DEBUG:
    debug_config = dict(config)
    debug_config.update({'N_TRAIN': 500, 'N_TEST': 200, 'N_RANDOM_PERMUTATIONS': 2})
    debug_ckpt_tmpl = CFG.CHECKPOINT_PATH_TEMPLATE.replace('.pkl', '_debug.pkl')

    if FORCE_RERUN_DEBUG:
        for alpha in [0.0, 0.5]:
            stale_ckpt = debug_ckpt_tmpl.format(alpha=alpha)
            if os.path.exists(stale_ckpt):
                os.remove(stale_ckpt)
                print(f'Removed stale debug checkpoint: {stale_ckpt}')

    debug_results = run_outer_seed_loop_synthetic(
        outer_seeds=[0, 1],
        alpha_values=[0.0, 0.5],
        config=debug_config,
        w_cols=w_cols,
        checkpoint_path_template=debug_ckpt_tmpl,
    )
    print('Debug run complete!')
    for alpha in [0.0, 0.5]:
        for seed in [0, 1]:
            m = debug_results[alpha][seed]['metrics'].get('CDV_SEPARATE', {}).get('DR_RF', {})
            print(f'  α={alpha}, seed={seed}: CDV_SEPARATE DR_RF ATE MSE = {m.get("ate_mse", "N/A")}')
else:
    print('Debug run skipped.')

## 4. Full Experiment — Outer Seed × Alpha Loop

In [ ]:
FORCE_RERUN_FULL = False  # delete stale full-run checkpoints before running

if FORCE_RERUN_FULL:
    for alpha in CFG.ALPHA_VALUES:
        stale_ckpt = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
        if os.path.exists(stale_ckpt):
            os.remove(stale_ckpt)
            print(f'Removed stale checkpoint: {stale_ckpt}')

print(f'Starting full experiment: {CFG.N_OUTER_SEEDS} seeds × {len(CFG.ALPHA_VALUES)} alphas.')
print(f'Checkpoint template: {CFG.CHECKPOINT_PATH_TEMPLATE}')
print('Resumes automatically from last completed (seed, alpha) pair.\n')

results_by_alpha = run_outer_seed_loop_synthetic(
    outer_seeds=CFG.OUTER_SEEDS,
    alpha_values=CFG.ALPHA_VALUES,
    config=config,
    w_cols=w_cols,
    checkpoint_path_template=CFG.CHECKPOINT_PATH_TEMPLATE,
)

print('\nExperiment complete!')
for alpha in CFG.ALPHA_VALUES:
    n = len(results_by_alpha.get(alpha, {}))
    print(f'  alpha={alpha:.2f}: {n} seeds')

## 5. Summary

In [ ]:
# Quick preview: DR_RF ATE MSE for GLOBAL_SENTINEL vs CDV_SEPARATE per alpha
print(f'Quick metric preview (DR_RF, mean ATE MSE):')
print(f'{"alpha":<8} {"GLOBAL_SENTINEL":<18} {"CDV_SEPARATE"}')
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    gs_vals  = [sr['metrics'].get('GLOBAL_SENTINEL', {}).get('DR_RF', {}).get('ate_mse', np.nan) for sr in res.values()]
    cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get('DR_RF', {}).get('ate_mse', np.nan) for sr in res.values()]
    print(f'{alpha:<8.2f} {np.nanmean(gs_vals):<18.5f} {np.nanmean(cdv_vals):.5f}')